# Tutorial 07 — Governance and Anomaly Detection

**No API key required. Fully deterministic.**

In a multi-tenant deployment, some tenants may behave abnormally — excessive rejection rates,
runaway cost utilisation, or spinning up too many concurrent jobs. eXo-brain provides two
independent governance layers to handle this:

1. **`detect_governance_anomalies`** — advisory-only detector; flags metrics that exceed
   configured thresholds without blocking any operation.
2. **`ByocFairAdmissionCoordinator`** — deterministic **process-local** admission control with a **global inflight cap**
   and grant ordering when slots free up (starvation-resistant under contention;
   see `tests/modules/policies/test_byoc_fairness.py`). **This notebook demonstrates:** cap,
   timeout-as-`None`, release, and a blocking waiter woken by `release()` — not every fairness edge case.

Both are independent of the ingress gate chain from Tutorial 03.

In [1]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
try:
    from dotenv import load_dotenv
    load_dotenv(_root / ".env", override=False)
except ImportError:
    pass

## Part 1 — BYOC governance model

**BYOC** (Bring Your Own Compute) means customers use shared eXo-brain infrastructure
with their own configuration. Without governance:
- One tenant's runaway usage can starve others
- Silent rejection spikes go unnoticed
- Cost budgets are exceeded before anyone reacts

The two governance tools are complementary:
- Anomaly detector: **"something is wrong — take a look"**
- Admission coordinator: **"global inflight full — wait or time out"**

## Part 2 — Simulate 3 tenants

We define metric snapshots for three tenants:
- `tenant-a` — healthy usage
- `tenant-b` — healthy usage, slightly higher rejection rate
- `tenant-c` — anomalous: near-maximum cost utilisation and very high rejection rate

In [2]:
from src.policies.governance_anomaly_detector import (
    detect_governance_anomalies,
    GovernanceAnomalyThresholds,
    GovernanceAnomaly,
)

# Shared thresholds for all tenants
thresholds = GovernanceAnomalyThresholds(
    cost_utilization_threshold=0.9,   # flag if > 90% of cost budget used
    rejection_rate_threshold=0.2,     # flag if > 20% of turns rejected
    reason_share_threshold=0.6,       # flag if one rejection reason > 60% of all rejections
    min_submit_attempts=5,
    min_rejection_count=3,
)

tenant_metrics = {
    "tenant-a": {
        "cost_utilization_ratio": 0.45,
        "rejection_rate": 0.05,
        "submit_attempts_total": 100,
        "rejected_results_total": 5,
        "rejection_reason_counts": {"POLICY_BLOCKED": 2, "TIMEOUT": 2, "RATE_LIMIT": 1},
    },
    "tenant-b": {
        "cost_utilization_ratio": 0.60,
        "rejection_rate": 0.18,
        "submit_attempts_total": 80,
        "rejected_results_total": 14,
        "rejection_reason_counts": {"POLICY_BLOCKED": 6, "TIMEOUT": 5, "RATE_LIMIT": 3},
    },
    "tenant-c": {
        "cost_utilization_ratio": 0.95,  # above threshold
        "rejection_rate": 0.90,          # well above threshold
        "submit_attempts_total": 200,
        "rejected_results_total": 180,
        "rejection_reason_counts": {"POLICY_BLOCKED": 160, "TIMEOUT": 20},
    },
}

assert thresholds.cost_utilization_threshold == 0.9
assert thresholds.rejection_rate_threshold == 0.2
assert thresholds.reason_share_threshold == 0.6
assert set(tenant_metrics) == {"tenant-a", "tenant-b", "tenant-c"}
assert tenant_metrics["tenant-c"]["cost_utilization_ratio"] == 0.95
assert tenant_metrics["tenant-c"]["rejection_rate"] == 0.90

print("Tenant metrics loaded for:", list(tenant_metrics.keys()))

Tenant metrics loaded for: ['tenant-a', 'tenant-b', 'tenant-c']


## Part 3 — Run anomaly detection

`detect_governance_anomalies` is a pure function — no side effects, no blocking.
It returns a list of `GovernanceAnomaly` findings (empty list = healthy).

In [3]:
a_anomalies = detect_governance_anomalies(
    cost_utilization_ratio=tenant_metrics["tenant-a"]["cost_utilization_ratio"],
    rejection_rate=tenant_metrics["tenant-a"]["rejection_rate"],
    submit_attempts_total=tenant_metrics["tenant-a"]["submit_attempts_total"],
    rejected_results_total=tenant_metrics["tenant-a"]["rejected_results_total"],
    rejection_reason_counts=tenant_metrics["tenant-a"]["rejection_reason_counts"],
    thresholds=thresholds,
)
b_anomalies = detect_governance_anomalies(
    cost_utilization_ratio=tenant_metrics["tenant-b"]["cost_utilization_ratio"],
    rejection_rate=tenant_metrics["tenant-b"]["rejection_rate"],
    submit_attempts_total=tenant_metrics["tenant-b"]["submit_attempts_total"],
    rejected_results_total=tenant_metrics["tenant-b"]["rejected_results_total"],
    rejection_reason_counts=tenant_metrics["tenant-b"]["rejection_reason_counts"],
    thresholds=thresholds,
)
c_anomalies = detect_governance_anomalies(
    cost_utilization_ratio=tenant_metrics["tenant-c"]["cost_utilization_ratio"],
    rejection_rate=tenant_metrics["tenant-c"]["rejection_rate"],
    submit_attempts_total=tenant_metrics["tenant-c"]["submit_attempts_total"],
    rejected_results_total=tenant_metrics["tenant-c"]["rejected_results_total"],
    rejection_reason_counts=tenant_metrics["tenant-c"]["rejection_reason_counts"],
    thresholds=thresholds,
)

for tenant_id, anomalies in (
    ("tenant-a", a_anomalies),
    ("tenant-b", b_anomalies),
    ("tenant-c", c_anomalies),
):
    print(f"\n{tenant_id}: {len(anomalies)} anomaly/ies")
    for a in anomalies:
        print(f"  code      : {a.code}")
        print(f"  severity  : {a.severity}")
        print(f"  message   : {a.message}")
        print(f"  value     : {a.value:.2f}  threshold: {a.threshold:.2f}")

assert a_anomalies == []
assert b_anomalies == []

c_codes = [a.code for a in c_anomalies]
assert c_codes == [
    "BYOC_COST_UTILIZATION_SPIKE",
    "BYOC_REJECTION_RATE_SPIKE",
    "BYOC_REJECTION_REASON_DOMINANCE",
]

dominance = next(a for a in c_anomalies if a.code == "BYOC_REJECTION_REASON_DOMINANCE")
assert dominance.reason_code == "POLICY_BLOCKED"
assert round(dominance.value, 2) == 0.89
assert dominance.threshold == 0.6
assert all(a.severity == "warning" for a in c_anomalies)

edge = detect_governance_anomalies(
    cost_utilization_ratio=0.9,
    rejection_rate=0.2,
    submit_attempts_total=5,
    rejected_results_total=3,
    rejection_reason_counts={"X": 3},
    thresholds=thresholds,
)
assert [a.code for a in edge] == [
    "BYOC_COST_UTILIZATION_SPIKE",
    "BYOC_REJECTION_RATE_SPIKE",
    "BYOC_REJECTION_REASON_DOMINANCE",
]

print("\nPASS — anomaly detection: tenant-a/b clean, tenant-c flagged with 3 codes")


tenant-a: 0 anomaly/ies

tenant-b: 0 anomaly/ies

tenant-c: 3 anomaly/ies
  code      : BYOC_COST_UTILIZATION_SPIKE
  severity  : warning
  message   : Tenant cost utilization exceeded advisory threshold.
  value     : 0.95  threshold: 0.90
  code      : BYOC_REJECTION_RATE_SPIKE
  severity  : warning
  message   : Tenant rejection rate exceeded advisory threshold.
  value     : 0.90  threshold: 0.20
  code      : BYOC_REJECTION_REASON_DOMINANCE
  severity  : warning
  message   : A single rejection reason dominates tenant failures.
  value     : 0.89  threshold: 0.60

PASS — anomaly detection: tenant-a/b clean, tenant-c flagged with 3 codes


## Part 4 — Fair admission: global inflight cap

`ByocFairAdmissionCoordinator(max_inflight_global=3)` allows at most 3 concurrent inflight
requests **across all tenants combined**. The 4th `acquire()` waits up to `wait_timeout_ms` and
returns **`None`** if no slot opens in time (admission timeout — not an exception).

In [4]:
import threading
import time
from src.policies.byoc_fairness import ByocFairAdmissionCoordinator, FairAdmissionToken

coordinator = ByocFairAdmissionCoordinator(max_inflight_global=3)

# Acquire 3 slots — all should succeed
token_a = coordinator.acquire(tenant_id="tenant-a", wait_timeout_ms=100)
token_b = coordinator.acquire(tenant_id="tenant-b", wait_timeout_ms=100)
token_c = coordinator.acquire(tenant_id="tenant-c", wait_timeout_ms=100)

print("token_a:", token_a)
print("token_b:", token_b)
print("token_c:", token_c)

# 4th acquire — no slots available, times out → returns None
token_d = coordinator.acquire(tenant_id="tenant-a", wait_timeout_ms=50)
print("token_d (should be None):", token_d)

assert token_a == FairAdmissionToken(tenant_id="tenant-a", request_id=1)
assert token_b == FairAdmissionToken(tenant_id="tenant-b", request_id=2)
assert token_c == FairAdmissionToken(tenant_id="tenant-c", request_id=3)
assert token_d is None

stats_after_cap = coordinator.stats()
assert stats_after_cap == {
    "fair_admission_max_inflight_global": 3,
    "fair_admission_inflight_total": 3,
    "fair_admission_pending_total": 0,
}

print("\nPASS — 3 slots granted, 4th timed out correctly")

token_a: FairAdmissionToken(tenant_id='tenant-a', request_id=1)
token_b: FairAdmissionToken(tenant_id='tenant-b', request_id=2)
token_c: FairAdmissionToken(tenant_id='tenant-c', request_id=3)
token_d (should be None): None

PASS — 3 slots granted, 4th timed out correctly


## Part 5 — Inspect admission stats

In [5]:
stats = coordinator.stats()
print("Admission stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")

assert stats == {
    "fair_admission_max_inflight_global": 3,
    "fair_admission_inflight_total": 3,
    "fair_admission_pending_total": 0,
}
print("\nPASS — stats reflect 3 inflight slots taken")

Admission stats:
  fair_admission_max_inflight_global: 3
  fair_admission_inflight_total: 3
  fair_admission_pending_total: 0

PASS — stats reflect 3 inflight slots taken


## Part 6 — Release frees a slot (same coordinator)

Releasing a token frees one global inflight slot. A **subsequent** `acquire()` on the main
thread can succeed immediately when a slot is free.

Part 6b (below) proves a **background thread** blocked in `acquire()` is woken when a slot is released.

In [6]:
# Release one token — slot becomes available
assert token_a is not None
assert token_b is not None
assert token_c is not None
coordinator.release(token_a)
print("Released token_a")

stats_after = coordinator.stats()
print("Stats after release:")
for k, v in stats_after.items():
    print(f"  {k}: {v}")

assert stats_after == {
    "fair_admission_max_inflight_global": 3,
    "fair_admission_inflight_total": 2,
    "fair_admission_pending_total": 0,
}

# Subsequent acquire on this thread (not a background waiter)
token_e = coordinator.acquire(tenant_id="tenant-b", wait_timeout_ms=100)
print("token_e after release:", token_e)
assert token_e is not None
assert token_e == FairAdmissionToken(tenant_id="tenant-b", request_id=5)

# Clean up remaining tokens from Parts 4–6
coordinator.release(token_b)
coordinator.release(token_c)
coordinator.release(token_e)

assert coordinator.stats()["fair_admission_inflight_total"] == 0
assert coordinator.stats()["fair_admission_pending_total"] == 0

print("\nPASS — release frees a slot; subsequent acquire succeeds")

Released token_a
Stats after release:
  fair_admission_max_inflight_global: 3
  fair_admission_inflight_total: 2
  fair_admission_pending_total: 0
token_e after release: FairAdmissionToken(tenant_id='tenant-b', request_id=5)

PASS — release frees a slot; subsequent acquire succeeds


## Part 6b — Background waiter wakes on release

With `max_inflight_global=1`, one holder blocks the only slot. A second `acquire()` on another
thread waits until `release()` frees the slot — then the waiter receives a token.

In [7]:
coordinator_wait = ByocFairAdmissionCoordinator(max_inflight_global=1)
holder = coordinator_wait.acquire(tenant_id="tenant-hold", wait_timeout_ms=200)
assert holder is not None, "holder should take the only slot"

waiter_box: dict[str, object] = {}
waiter_started = threading.Event()

def _waiter_acquire() -> None:
    waiter_started.set()
    tok = coordinator_wait.acquire(tenant_id="tenant-wait", wait_timeout_ms=3000)
    waiter_box["token"] = tok

waiter_thread = threading.Thread(target=_waiter_acquire, daemon=True)
waiter_thread.start()
assert waiter_started.wait(timeout=1.0), "waiter thread should start acquire"
time.sleep(0.05)  # allow waiter to block on the condition variable
assert waiter_thread.is_alive(), "waiter should still be blocked while slot is held"
assert waiter_box.get("token") is None, "no token yet while slot is held"

coordinator_wait.release(holder)
waiter_thread.join(timeout=2.0)
assert not waiter_thread.is_alive(), "waiter thread should finish"
assert waiter_box.get("token") is not None, "release should grant the waiting acquire"

waiter_token = waiter_box["token"]
assert isinstance(waiter_token, FairAdmissionToken)
assert waiter_token.tenant_id == "tenant-wait"

coordinator_wait.release(waiter_token)
assert coordinator_wait.stats()["fair_admission_inflight_total"] == 0
assert coordinator_wait.stats()["fair_admission_pending_total"] == 0

print("\nPASS — background waiter received token after release")


PASS — background waiter received token after release


## Part 7 — Per-tenant policy overlays

`TenantPolicyOverlayStore` maps tenant IDs to their policy configuration overlays.
This is the mechanism by which different tenants can have different ingress profiles,
classifier settings, and custom rules — stored per tenant under independent tenant keys.
`get_overlay()` returns a copy so callers cannot mutate stored overlays accidentally.

In [8]:
from src.tenancy.policy_overlay import TenantPolicyOverlayStore

overlay_store = TenantPolicyOverlayStore()

# Each tenant brings their own policy configuration
overlay_store.set_overlay("tenant-a", {
    "ingress_profile": "baseline",
    "ingress_classifier_mode": "off",
})

overlay_store.set_overlay("tenant-b", {
    "ingress_profile": "strict",
    "ingress_classifier_mode": "shadow",
    "ingress_classifier_threshold": 0.65,
})

overlay_store.set_overlay("tenant-c", {
    "ingress_profile": "hardened",
    "ingress_classifier_mode": "enforce",
    "ingress_classifier_threshold": 0.5,
    "ingress_custom_rules": [
        {
            "rule_id":    "c-block-001",
            "action":     "deny",
            "match_type": "contains_any",
            "patterns":   ["export all", "bypass limit"],
            "reason_code": "TENANT_C_BLOCKED",
            "message":    "This action is not permitted for your account.",
        }
    ],
})

for tid in ["tenant-a", "tenant-b", "tenant-c"]:
    overlay = overlay_store.get_overlay(tid)
    print(f"{tid}: profile={overlay.get('ingress_profile')}, "
          f"classifier={overlay.get('ingress_classifier_mode')}")

a = overlay_store.get_overlay("tenant-a")
b = overlay_store.get_overlay("tenant-b")
c = overlay_store.get_overlay("tenant-c")

assert a["ingress_profile"] == "baseline"
assert a["ingress_classifier_mode"] == "off"
assert b["ingress_profile"] == "strict"
assert b["ingress_classifier_mode"] == "shadow"
assert b["ingress_classifier_threshold"] == 0.65
assert c["ingress_profile"] == "hardened"
assert c["ingress_classifier_mode"] == "enforce"
assert c["ingress_classifier_threshold"] == 0.5
assert c["ingress_custom_rules"][0]["reason_code"] == "TENANT_C_BLOCKED"

retrieved = overlay_store.get_overlay("tenant-c")
retrieved["ingress_profile"] = "mutated"
assert overlay_store.get_overlay("tenant-c")["ingress_profile"] == "hardened"

print("\nPASS — per-tenant overlays stored and retrieved independently")

tenant-a: profile=baseline, classifier=off
tenant-b: profile=strict, classifier=shadow
tenant-c: profile=hardened, classifier=enforce

PASS — per-tenant overlays stored and retrieved independently


## Summary

| Capability | Module | Key API |
|---|---|---|
| Anomaly detection | `src/policies/governance_anomaly_detector` | `detect_governance_anomalies(...)` |
| Anomaly thresholds | `src/policies/governance_anomaly_detector` | `GovernanceAnomalyThresholds` |
| Anomaly finding | `src/policies/governance_anomaly_detector` | `GovernanceAnomaly.code/severity/value/threshold` |
| Fair admission (global cap) | `src/policies/byoc_fairness` | `ByocFairAdmissionCoordinator.acquire()` → `None` on timeout |
| Admission token / release | `src/policies/byoc_fairness` | `FairAdmissionToken`, `release()` |
| Admission stats | `src/policies/byoc_fairness` | `coordinator.stats()` |
| Grant ordering under contention | `tests/modules/policies/test_byoc_fairness.py` | covered by unit tests; not asserted in this notebook |
| Per-tenant config | `src/tenancy/policy_overlay` | `TenantPolicyOverlayStore.set_overlay()` |

**Key insight:** Anomaly detection is advisory — it never blocks. Fair admission is deterministic:
when the global inflight cap is full, `acquire()` times out and returns `None`; `release()` frees
a slot and can wake a blocked waiter. Grant ordering across tenants is implemented in the coordinator
and covered by unit tests — this notebook focuses on cap, timeout, release, and the waiter wake-up.
Both layers are independent of the ingress gate chain.

## Notebook navigation

| If you want… | Open |
|---|---|
| Previous / next in learning path | See `notebooks/README.md` index |
| Fast module smoke after a code change | `check_01` … `check_04` |
| Ingress or tool boundary proofs | `edge_01`, `edge_02` |
| Local governance lab (no API key) | `tutorial_08_governed_execution_sandbox.ipynb` |
| Live governance contrasts (optional API key) | `tutorial_09_governed_execution_live.ipynb` |
| Evaluator time-boxed paths | `notebooks/EVALUATOR_GUIDE.md` |

**Regenerate notebooks:** edit this build script, then `python notebooks/build_tutorials.py` (do not hand-edit `.ipynb` JSON).